# Online Retail Store Analysis

## Preparing environment

### Loading up needed modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from pathlib import Path

### Importing data

In [ ]:
df = pd.read_csv(Path("datasets") / "online_retail.csv")
df.info()

In [ ]:
df.sample(10)

# Data preparation

## Checking column completeness

In [ ]:
all_cols = df.columns
bar_colors = []
counted_cols = []
for col_name in all_cols:
    completeness = 1 - len(df.loc[df[col_name].isnull(), col_name]) / len(df)
    counted_cols.append(completeness)
    if completeness == 1.0:
        bar_colors.append("darkgreen")
    elif completeness > 0.8:
        bar_colors.append("gold")
    else:
        bar_colors.append("red")


def percentage_format(x, position):
    return f"{round(x * 100)}%"


fig, axs = plt.subplots(figsize=(len(counted_cols) * 2, 5))
axs.set_title("Data completeness %")
axs.yaxis.set_major_formatter(percentage_format)
axs.bar(all_cols, counted_cols, color=bar_colors)
axs.set_ylim(0, 1)
plt.show()

### Missing Customer Data

As we can see, the dataset lacks `CustomerID` for many records. By dropping these rows, this analysis excludes "guest checkouts". Our focus is strictly on registered users, allowing for a genuine evaluation of customer loyalty, retention metrics, and long-term user behavior.

### Data Architecture: Splitting the Dataset

To ensure absolute metric accuracy, we must branch our dataset based on the business question being asked:

1. **Net Revenue & LTV (Full Dataset):** The original dataset includes invoices starting with "C" (cancellations and returns). These are crucial for calculating true Net Lifetime Value (LTV), as returns offset gross revenue.
2. **Cohort Retention (Filtered Dataset):** For analyzing user loyalty, we create a separate `df_no_cancel` dataframe excluding cancellations. If we kept returns here, our grouping logic would register a product return in a subsequent month as a new "active shopping session," artificially inflating our retention rates.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [ ]:
df_no_cancel = df.dropna(subset="CustomerID")

# Removing cancelled invoices and returned products
df_no_cancel = df_no_cancel[~df_no_cancel["InvoiceNo"].str.startswith("C")]
df_no_cancel = df_no_cancel[
    (df_no_cancel["Quantity"] > 0) & (df_no_cancel["UnitPrice"] > 0)
]
df_no_cancel.head()

## Cohort retention analysis

Here I am assigning new columns to the retention data frame:
- `InvoiceMonth` is as the name says - month an invoice was issued.
- `CohortMonth` is based on the newly created `InvoiceMonth` column - it assigns every customer to the first month they made a purchase in our store.
- `CohortIndex` is the difference in months between the first purchase and later purchases.

In [ ]:
df_retention = df_no_cancel.assign(
    InvoiceMonth=lambda x: x["InvoiceDate"].values.astype("datetime64[M]"),
    CohortMonth=lambda x: x.groupby("CustomerID")["InvoiceMonth"].transform("min"),
    CohortIndex=lambda x: (
        (x["InvoiceMonth"].dt.year - x["CohortMonth"].dt.year) * 12
        + (x["InvoiceMonth"].dt.month - x["CohortMonth"].dt.month)
    ),
)
df_retention.head()

Group by cohort months (First purchases of customers)

In [ ]:
cohort_data = (
    df_retention.groupby(["CohortMonth", "CohortIndex"])
    .agg({"CustomerID": "nunique"})
    .reset_index()
)
cohort_data

Creating the table

In [ ]:
cohort_table = pd.pivot(
    cohort_data, columns="CohortIndex", values="CustomerID", index="CohortMonth"
)
cohort_table

### Creating the heatmap using Seaborn

Basic heatmap already shows some interesing information - last year's December attracted almost twice as many customers as any other month later on.

However, the current heatmap isn't very readable. Better approach to showing customer retention would be to use percentage instead of raw numbers.

In [ ]:
y_labels = cohort_table.index.strftime("%B %Y")

plt.figure(figsize=(16, 8))
sns.heatmap(cohort_table, cmap="Blues", yticklabels=y_labels, fmt="g", annot=True)
plt.title("Customer retention Heatmap")
plt.show()

#### Improving heatmap readability
Here I am dividing whole rows by the first element of each. This way I can measure retention in percentage of consumers who stayed.

In [ ]:
cohort_table_ratio = cohort_table.divide(cohort_table.iloc[:, 0], axis=0)
cohort_table_ratio

#### Readable heatmap
The heatmap now is much more readable, but now we lost the key information of an absolute number of customers acquired that month.

In [ ]:
y_labels = cohort_table_ratio.index.strftime("%B %Y")
plt.figure(figsize=(16, 8))
sns.heatmap(
    cohort_table_ratio.iloc[:, 1:],
    cmap="Blues",
    yticklabels=y_labels,
    annot=True,
    fmt=".0%",
)
plt.ylabel("First purchase month")
plt.xlabel("Months since the first purchase")
plt.title("Customer retention analysis (Cohorts)")
plt.show()

#### Adding the cohort size
To address that problem we need to create a subplot in which the absolute number of customers would be visible.

In [ ]:
cohort_size = (
    df_retention.groupby("CohortMonth")
    .agg({"CustomerID": "nunique"})
    .rename(columns={"CustomerID": "Cohort Size"})
)

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 10), sharey=True, gridspec_kw={"width_ratios": [1, 14]}
)
sns.heatmap(cohort_size, cbar=False, cmap="Blues", annot=True, fmt="g", ax=ax1)
sns.heatmap(
    cohort_table_ratio.iloc[:, 1:],
    cmap="Blues",
    annot=True,
    fmt=".0%",
    ax=ax2,
    yticklabels=y_labels,
)
plt.subplots_adjust(wspace=0.02)
ax1.set_ylabel("First purchase month", fontsize=12)
ax2.set_ylabel("")
ax2.set_xlabel("Months since the first purchase", fontsize=12)
ax2.set_title("Customer retention analysis (Cohorts)")
plt.show()

### Executive Summary & Business Recommendations

#### Missing Data & Cohort Bias
* **Observation:** The great results of the December 2010 cohort are affected by missing past data. Because our dataset starts in this month, this group probably contains many old, loyal customers, not just the new ones. The sudden drop in December 2011 is also caused by incomplete data for that month.
* **Recommendation:** To correctly measure customer retention, we need to get historical data from before December 2010. This will help us separate the real "new" customers from the "existing" ones.

#### Seasonality & Marketing Budget
* **Observation:** There is a clear drop in new customer acquisition and their loyalty during the summer months. On the other hand, the Fall season (September-November) shows strong acquisition and high retention rates.
* **Recommendation:** The marketing department should adjust its budget. We should spend less money during the weak summer months and move this budget to September and October. This will help build a strong customer base before the winter holidays.

#### The "November Comeback"
* **Observation:** A diagonal pattern on the retention matrix shows that many inactive customers from previous months came back in November 2011. This was probably driven by Black Friday and early Christmas shopping.
* **Recommendation:** We should take advantage of this predictable trend. Product and Marketing teams should plan automated "win-back" email campaigns for inactive customers from Spring and Summer.


## RFM Analysis

In [ ]:
df_monetary = (
    df.assign(TotalPrice=lambda x: x["UnitPrice"] * x["Quantity"])
    .groupby("CustomerID")
    .agg(Monetary=("TotalPrice", "sum"))
)

ref_date = df_no_cancel["InvoiceDate"].max() + pd.Timedelta(days=1)
df_rf = (
    df_no_cancel.assign(
        RecencyDays=lambda x: (ref_date - x["InvoiceDate"]).dt.days,
    )
    .groupby("CustomerID")
    .agg(Recency=("RecencyDays", "min"), Frequency=("InvoiceNo", "nunique"))
)
df_rfm = df_rf.join(df_monetary)
df_rfm = df_rfm.set_index(df_rfm.index.astype("int64"))
df_rfm = df_rfm.loc[df_rfm["Monetary"] > 0].copy()
df_rfm.sample(5)

Assign points to customers

In [ ]:
labels = [1, 2, 3, 4, 5]
labels_reversed = labels.copy()
labels_reversed.reverse()

rfm_score = df_rfm.sort_values(
    by=["Monetary", "Frequency"], ascending=[True, True]
).assign(
    RecScore=lambda x: pd.qcut(
        x["Recency"].rank(method="first"), q=5, labels=labels_reversed
    ),
    FreqScore=lambda x: pd.qcut(
        x["Frequency"].rank(method="first"), q=5, labels=labels
    ),
    MonScore=lambda x: pd.qcut(x["Monetary"].rank(method="first"), q=5, labels=labels),
    RFMScore=lambda x: (
        x["RecScore"].astype("str")
        + "-"
        + x["FreqScore"].astype("str")
        + "-"
        + x["MonScore"].astype("str")
    ),
)
rfm_score

In [ ]:
r = rfm_score["RecScore"].astype(int)
f = rfm_score["FreqScore"].astype(int)
m = rfm_score["MonScore"].astype(int)

conditions = [
    (r + f + m) >= 14,
    (r <= 2) & (f >= 4),
    (r >= 4) & (f <= 2),
    (r <= 2) & (m <= 2),
]
choices = ["Champion", "At risk", "Newcomer", "Hibernating"]
rfm_score = rfm_score.assign(
    CustomerSegment=pd.Categorical(np.select(conditions, choices, default="Standard"))
)
sns.barplot(rfm_score.groupby("CustomerSegment").size())
plt.show()

Pareto rule in gross revenue per customer

In [ ]:
df_pareto = (
    df.assign(TotalPrice=lambda x: x["UnitPrice"] * x["Quantity"])
    .groupby("CustomerID")
    .agg(TotalSpending=("TotalPrice", "sum"))
    .sort_values(by="TotalSpending", ascending=False)
    .assign(
        SpendingCumSum=lambda x: x["TotalSpending"].cumsum(),
        RevenuePercent=lambda x: x["SpendingCumSum"] / x["TotalSpending"].sum(),
    )
)

top_80p_revenue = df_pareto[df_pareto["RevenuePercent"] <= 0.80]

top_count = len(top_80p_revenue)
total_count = len(df_pareto)

ratio = top_count / total_count

f"80% of revenue is generated by top {ratio:.1%} customers"

### Hourly buying time in span of a week

In [ ]:
days_dict = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}
days_order = np.arange(0, 7)
summary_weekly_hourly = (
    df.assign(
        Day=lambda x: x["InvoiceDate"].dt.day_of_week,
        Hour=lambda x: x["InvoiceDate"].dt.hour,
    )
    .groupby(["Day", "Hour"])
    .agg(InvoiceCount=("InvoiceNo", "nunique"))
    .reset_index()
)

weekly_purchase_matrix = (
    summary_weekly_hourly.pivot(index="Day", columns="Hour", values="InvoiceCount")
    .reindex(days_order)
    .fillna(0)
)

golden_hour_row = summary_weekly_hourly.loc[
    summary_weekly_hourly["InvoiceCount"].idxmax()
]

print(
    f"Our golden hour is at {days_dict[golden_hour_row['Day']]}, {golden_hour_row['Hour']}:00. {golden_hour_row['InvoiceCount']} transactions were made during that hour."
)

plt.figure(figsize=(16, 8))
ax = sns.heatmap(weekly_purchase_matrix, annot=True, cmap="YlGnBu", fmt="g")
ax.set_yticklabels(days_dict.values(), rotation=0)
plt.title("When to send the push? (Purchases by hours and days of week)")
plt.ylabel("Day of week", fontsize=12)
plt.xlabel("Hour of purchase", fontsize=12)
plt.show()

Products bought in pairs

In [ ]:
product_map = df[["StockCode", "Description"]].drop_duplicates(
    subset=["StockCode"], keep="first"
)
df_pairs = df[["InvoiceNo", "StockCode"]]
df_pairs = df_pairs.merge(df_pairs, on="InvoiceNo", suffixes=("_A", "_B"))
df_pairs

In [ ]:
df_pairs = df_pairs[df_pairs["StockCode_A"] < df_pairs["StockCode_B"]]
df_pairs = (
    df_pairs.groupby(
        [
            "StockCode_A",
            "StockCode_B",
        ]
    )
    .agg({"StockCode_B": "count"})
    .rename(columns={"StockCode_B": "Count"})
    .reset_index()
)

In [ ]:
df_pairs_pretty = (
    df_pairs.merge(product_map, left_on="StockCode_A", right_on="StockCode")
    .drop(columns="StockCode")
    .rename(columns={"Description": "Description_A"})
)
df_pairs_pretty = (
    df_pairs_pretty.merge(product_map, left_on="StockCode_B", right_on="StockCode")
    .drop(columns="StockCode")
    .rename(columns={"Description": "Description_B"})
)
df_pairs_pretty = df_pairs_pretty[
    ["StockCode_A", "Description_A", "StockCode_B", "Description_B", "Count"]
].sort_values(by="Count", ascending=False)
df_pairs_pretty.head(20)

Top Countries by average revenue per user

In [ ]:
arpu_summary = (
    df[df["Country"] != "United Kingdom"]
    .groupby(["Country", "CustomerID"])
    .agg({"TotalPrice": "sum"})
    .reset_index()
    .groupby("Country")
    .filter(lambda x: len(x) >= 20)
    .groupby("Country")
    .agg({"CustomerID": "nunique", "TotalPrice": "mean"})
    .rename(columns={"TotalPrice": "ARPU", "CustomerID": "Customers"})
)
arpu_summary

Median spending per Cohort month

In [ ]:
ltv = df.groupby(["CohortMonth", "CustomerID"]).agg({"TotalPrice": "sum"}).reset_index()
ltv.groupby("CohortMonth").agg(MedianSpending=("TotalPrice", "median"))

Churn analysis

In [ ]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
customer_last_purchase = df.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max")
)
customer_last_purchase["Recency"] = (
    reference_date - customer_last_purchase["LastPurchase"]
).dt.days
customer_last_purchase["Status"] = np.where(
    customer_last_purchase["Recency"] < 90, "Active", "Churned"
)
customer_last_purchase.index = customer_last_purchase.index.astype("int64")
display(
    customer_last_purchase["Status"].value_counts(normalize=True),
    customer_last_purchase,
)

Median monthly spend per customer

In [ ]:
cohort_month_map = df[["CustomerID", "CohortMonth"]].drop_duplicates(
    subset="CustomerID", keep="first"
)
aov_heatmap = (
    df.groupby(["CustomerID", "CohortIndex"])
    .agg(SpendingInMonth=("TotalPrice", "sum"))
    .reset_index()
    .merge(cohort_month_map, left_on="CustomerID", right_on="CustomerID")
    .pivot_table(
        columns="CohortIndex",
        index="CohortMonth",
        values="SpendingInMonth",
        aggfunc="median",
    )
)
plt.figure(figsize=(21, 10))
sns.heatmap(aov_heatmap, annot=True, cmap="Blues", fmt=".0f", yticklabels=y_labels)
plt.title("Median monthly spend per customer", fontsize=18)
plt.ylabel("First purchase month", fontsize=12)
plt.xlabel("Months since the first purchase", fontsize=12)